In [ ]:
####################################
#ENVIRONMENT SETUP

In [ ]:
#LIBRARIES

#system
import os
os.environ["HDF5_USE_FILE_LOCKING"] = "FALSE"
import sys

#math and array operations
import numpy as np
import pandas as pd
import math

#plotting
import matplotlib
# matplotlib.use("Agg") #UNCOMMENT IF PLOTTING WITHIN JUPYTER DOCUMENT
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import TwoSlopeNorm
from matplotlib.colors import ListedColormap, BoundaryNorm

import cartopy.crs as ccrs
import cartopy.feature as cfeature

#data classes
import xarray as xr
import h5py
import pickle 

#loading bar
from tqdm import tqdm

#dates
from datetime import datetime, timedelta

In [ ]:
#Importing DirectoryManager Class
sys.path.append(os.path.join("/glade/u/home/aroseman/Projects/Regional-MPAS-Project/Code/CodeFiles/","DataAnalysis"))
from CLASSES_Directories import DirectoryManager_Class

In [ ]:
DirectoryManager = DirectoryManager_Class()

codeType = os.path.join("DataAnalysis", "Observation_Data")
dataType = "RainfallSpatialMetrics"

outputDirectory = DirectoryManager.GetOutputDirectory(codeType, dataType)
outputPlottingDirectory = DirectoryManager.GetOutputPlottingDirectory(codeType, dataType)

In [ ]:
#Importing ModelData Class
sys.path.append(os.path.join(DirectoryManager.mainCodeDirectory,"DataAnalysis","MPAS_Model_Data"))
from CLASSES_ModelData import StructuredModelData_Class, DataOperator_Class

In [ ]:
#Importing ModelData Class
sys.path.append(os.path.join(DirectoryManager.mainCodeDirectory,"DataAnalysis","MPAS_Model_Data"))
from CLASSES_PlottingModelData import FigurePlotting_Class

In [ ]:
#Setup
Region = "TRACER"; Case = "WET"; spinup_hours = "0"
Region = "TRACER"; Case = "DIURNAL"; spinup_hours = "-5"

Region = "Hawaii"; Case = "WET"; spinup_hours = "12"
Region = "Hawaii"; Case = "TRADES"; spinup_hours = "24"

In [ ]:
#Load Model Directory Class
RunType = (Region,Case,"NSSL",spinup_hours)
ModelData_NSSL = StructuredModelData_Class(DirectoryManager.mainDirectory, DirectoryManager.scratchDirectory, RunType)

RunType = (Region,Case,"TEMPO",spinup_hours)
ModelData_TEMPO = StructuredModelData_Class(DirectoryManager.mainDirectory, DirectoryManager.scratchDirectory, RunType)

In [ ]:
#Importing ModelData Class
sys.path.append(os.path.join(DirectoryManager.mainCodeDirectory,"DataAnalysis"))
from CLASSES_DataSubsetting import DataSubsetting_Class

In [ ]:
#Importing Radar Classes
sys.path.append(os.path.join(DirectoryManager.mainCodeDirectory,"DataAnalysis","Observation_Data"))
from CLASSES_RadarDataLoading import RadarData_MRMS_Class, RadarObservationMask_Class

sys.path.append(os.path.join(DirectoryManager.mainCodeDirectory,"DataAnalysis"))
from CLASSES_RadarDataPlotting import RadarPlotting_Class

In [ ]:
#IMPORT FUNCTIONS
# --- Add your Functions folder to sys.path ---
import sys
path = os.path.join(DirectoryManager.mainCodeDirectory, 'Functions_2.0')
sys.path.append(path)


# --- Import all your function modules ---
import importlib
modules = [
    "AreaAverageFunctions",
    "ComputationFunctions",
    "DataFunctions",
    "DerivativeFunctions",
    "PlottingFunctions",
    "StatisticalFunctions",
]
for mod in modules:
    globals()[mod] = importlib.import_module(mod)        # import module itself
    globals().update(vars(globals()[mod]))              # import all functions into global namespace

In [ ]:
####################################
#CALCULATING FUNCTIONS

In [ ]:
# #Loading Radar Mask #decided not to use here
# RadarDataMask = RadarObservationMask_Class.LoadMaskData(DirectoryManager, ModelData_NSSL) 

In [ ]:
def SubsetData_Time(data,yearmonthday):
    data_T = data.sel(time=yearmonthday)
    # print(data_T.time) #testing
    return data_T

In [ ]:
# Functions for calculating SCOR and SRMSE for rain rate fields

def CalculateSCOR(S, O):
    """
    #S = Simulated
    #O = Observations
    #SCOR = Cov(S,O)/[SD(S)*SD(O)] = (1/n)*sum(S'*O')/[sqrt(sum(M'^2)/n)*sqrt(sum(O'^2)/n)]
    """
    # ensure common valid mask
    mask = (~S.isnull()) & (~O.isnull())
    S = S.where(mask)
    O = O.where(mask)

    S_prime = S - S.mean(skipna=True)
    O_prime = O - O.mean(skipna=True)

    numerator = (S_prime * O_prime).sum(skipna=True)
    denominator = (
        (S_prime**2).sum(skipna=True) *
        (O_prime**2).sum(skipna=True)
    )**0.5

    SCOR = numerator / denominator

    return SCOR

def CalculateSRMSE(S, O):
    """
    S = Simulated
    O = Observations
    SRMSE = sqrt((1/n) * sum((S - O)^2))
    """

    mask = (~S.isnull()) & (~O.isnull())
    diff = (S - O).where(mask)

    SRMSE = ((diff**2).mean(skipna=True))**0.5

    return SRMSE

In [ ]:

## CalculateFSS_scores

# Leeuwenburg, T., Loveday, N., Ebert, E. E., Cook, H., Khanarmuei, M., Taggart, R. J., Ramanathan, N., Carroll, M., Chong, S., Griffiths, A., & Sharples, J. (2024). 
# scores: A Python package for verifying and evaluating models and predictions with xarray. Journal of Open Source Software, 9(99), 6889. https://doi.org/10.21105/joss.06889
# https://scores.readthedocs.io/en/2.0.0/tutorials/Fractions_Skill_Score.html

# pip install scores
from scores.spatial import fss_2d_single_field
from scores.fast.fss.typing import FssComputeMethod

def CalculateFSS_scores(forecast,observation,threshold,window_size):
    compute_method = FssComputeMethod.NUMPY
    threshold_operator = np.greater_equal
    # threshold_operator = np.greater
    
    fs_score = fss_2d_single_field(
        forecast,
        observation,
        event_threshold=threshold,
        window_size=window_size,           # same interpretation as 'scale'
        threshold_operator=threshold_operator,
        compute_method=compute_method # default and fastest
    )
    return fs_score*100



In [ ]:
#DATA LOADING FOR MRMS QPE DATA

def CorrectSimulationDates(ModelData, simulationDates):
    if int(ModelData.spinup_hours) <= 0:
        # Convert first date to Timestamp
        first_date = pd.to_datetime(simulationDates[0])
        # Subtract one day
        prev_date = (first_date - pd.Timedelta(days=1)).strftime('%Y-%m-%d')
        # Prepend
        simulationDates = [prev_date] + simulationDates
    return simulationDates
    
def GetMRMS_QPE_DataDirectory(ModelData, product="MultiSensor_QPE_01H_Pass2_00.00"):
    simulationDates = ModelData.simulationDates
    simulationDates2 = CorrectSimulationDates(ModelData, simulationDates)

    inputDirectory = os.path.join(DirectoryManager.dataDirectory,
             f"Observation_Data/{ModelData.region}/MRMS_RadarData",
             f"{simulationDates2[0]}_{simulationDates2[-1]}",product)
    return inputDirectory
    
def ReadPrecipData_MRMS_QPE(ModelData, yearmonthdayhour, inputDirectory):
    yearmonthday = yearmonthdayhour[0:8]
    hour = yearmonthdayhour[8:]

    MRMS_region = "CONUS" if ModelData.region=="TRACER" else "HAWAII"
    inputPath = os.path.join(inputDirectory,f"MRMSQPE_{MRMS_region}_{ModelData.region}_{yearmonthday}-{hour}0000.nc")
    try:
        precipData = xr.open_dataset(inputPath)["MultiSensor_QPE_01H_Pass2_00.00"].isel(time=0)
    except:
        print(f"{inputPath} does not exist in MRMS data ==> skipping")
        precipData = None
    #Note: data is in mm units
    return precipData


def GetAccumulatedPrecipData_MRMS_SpatialMetrics(ModelData): 
    #getting inputDirectory
    inputDirectory = GetMRMS_QPE_DataDirectory(ModelData)
    print(f"reading from {inputDirectory}")

    #getting yearmonthdayhour list
    dt_list = [datetime.strptime(t, "%Y-%m-%d_%H.%M.%S") for t in ModelData.timeStrings]
    start = dt_list[0]
    target = start + timedelta(hours=1)# + timedelta(hours=13) #only using for rainfall histogram
    # print(f"start_time: {target}")
    
    # Find the entry closest to +12 hours
    closest = min(dt_list, key=lambda x: abs(x - target))
    
    closest_idx = dt_list.index(closest)
    times = ModelData.timeStrings[closest_idx:]
    yearmonthdayhours = sorted({t.replace('-', '').replace('_', '').replace('.', '')[:10] 
                                for t in times})

    precipData_T = None
    SCOR_List = []; SRMSE_List = []; timeList = []
    FSS_List_1 = []; FSS_List_2 = []
    for count, (yearmonthdayhour, t) in tqdm(
        enumerate(zip(yearmonthdayhours, times)),
        total=len(yearmonthdayhours)):
        # print(yearmonthdayhour)

        #time from one hour prior to fix find last hour accumulated precip
        t_m1 = (datetime.strptime(t, "%Y-%m-%d_%H.%M.%S") - timedelta(hours=1))
        t_m1 = t_m1.strftime("%Y-%m-%d_%H.%M.%S")

        #Get Model Accumulated Rain in Last Hour
        data_t = ModelData.GetDataTimestep(t, printout=False)
        Simulated_t = data_t['rainnc'] + data_t['rainc']
        data_tm1 = ModelData.GetDataTimestep(t_m1, printout=False)
        Simulated_tm1 = data_tm1['rainnc'] + data_tm1['rainc']
        Simulated = Simulated_t - Simulated_tm1
        Simulated = DataSubsetting_Class.SubsetDataRegion(Simulated, ModelData) # --- subset the region ---
        
        #Get Model Accumulated Rain in Last Hour
        Observed = ReadPrecipData_MRMS_QPE(ModelData,yearmonthdayhour,inputDirectory)
        if Observed is None:
            print(f"Missing MRMS data for {yearmonthdayhour}")
            continue
        Observed = Observed.assign_coords(
            longitude = Observed.longitude - 360
        ).sortby("latitude")
        Observed = DataSubsetting_Class.SubsetDataRegion(Observed, ModelData) # --- subset the region ---

        Simulated = Simulated.interp(latitude=Observed.latitude,longitude=Observed.longitude)

        # --- calculating spatial metricss ---
        SCOR = CalculateSCOR(S=Simulated, O=Observed)
        SRMSE = CalculateSRMSE(S=Simulated, O=Observed)

        # FSS using square neighborhood of length of 18 km (window) with 1-h AP thresholds of 0.01 and 0.5 in (0.254 and 12.7 mm)
        FSS_1 = CalculateFSS_scores(Simulated.data,Observed.data,threshold=0.25,window_size=(18,18))
        FSS_2 = CalculateFSS_scores(Simulated.data,Observed.data,threshold=1,window_size=(18,18))

        SCOR_List.append(SCOR); SRMSE_List.append(SRMSE); 
        FSS_List_1.append(FSS_1); FSS_List_2.append(FSS_2)
        timeList.append(yearmonthdayhour)

    timeList = [datetime.strptime(t, "%Y%m%d%H") for t in timeList]
    return np.array(SCOR_List),np.array(SRMSE_List),np.array(FSS_List_1), np.array(FSS_List_2), np.array(timeList)

def LoadOrRun_GetAccumulatedPrecipData_MRMS_SpatialMetrics(
    ModelData,
    overwrite=False
):

    os.makedirs(outputDirectory, exist_ok=True)

    fileName = (
        f"{ModelData.region}_"
        f"{ModelData.case}_"
        f"{ModelData.spinup_hours}hrs"
        f"{ModelData.mpType}_"
        f"MRMS_QPE_SpatialMetrics.nc"
    )

    filePath = os.path.join(outputDirectory, fileName)

    # --------------------------------------------------
    # LOAD
    # --------------------------------------------------
    if os.path.exists(filePath) and not overwrite:
        print("Loading MRMS spatial metrics from disk:")
        print(f"  {filePath}")

        with xr.open_dataset(filePath) as ds:
            SCOR = ds["SCOR"].values
            SRMSE = ds["SRMSE"].values
            FSS_1 = ds["FSS_1"].values
            FSS_2 = ds["FSS_2"].values
            time_np = ds["time"].values

        timeArray = np.array(
            [t.astype("datetime64[ms]").astype(datetime) for t in time_np],
            dtype=object
        )

        return SCOR,SRMSE, FSS_1,FSS_2, timeArray

    # --------------------------------------------------
    # RUN
    # --------------------------------------------------
    print("Computing MRMS spatial metrics")

    SCOR, SRMSE, FSS_1, FSS_2, timeArray = GetAccumulatedPrecipData_MRMS_SpatialMetrics(ModelData)

    # --------------------------------------------------
    # SAVE
    # --------------------------------------------------
    ds = xr.Dataset(
        data_vars={
            "SCOR": ("time", SCOR),
            "SRMSE": ("time", SRMSE),
            "FSS_1": ("time", FSS_1),
            "FSS_2": ("time", FSS_2),
        },
        coords={
            "time": np.array(timeArray, dtype="datetime64[ms]")
        }
    )

    ds["SCOR"].attrs["description"] = "Spatial correlation coefficient (model vs MRMS)"
    ds["SRMSE"].attrs["description"] = "Spatial root mean square error (model vs MRMS)"

    ds.attrs["dataset"] = "MRMS_QPE_SpatialMetrics"
    ds.attrs["region"] = ModelData.region
    ds.attrs["case"] = ModelData.case
    ds.attrs["spinup_hours"] = ModelData.spinup_hours

    ds.to_netcdf(filePath)

    print("Saved MRMS spatial metrics to disk:")
    print(f"  {filePath}")

    return SCOR,SRMSE, FSS_1,FSS_2, timeArray

In [ ]:
####################################
#RUNNING

In [ ]:
SCOR_NSSL,SRMSE_NSSL, FSS_1_NSSL,FSS_2_NSSL, timeList_NSSL = LoadOrRun_GetAccumulatedPrecipData_MRMS_SpatialMetrics(ModelData_NSSL)
SCOR_TEMPO, SRMSE_TEMPO, FSS_1_TEMPO,FSS_2_TEMPO, timeList_TEMPO = LoadOrRun_GetAccumulatedPrecipData_MRMS_SpatialMetrics(ModelData_TEMPO)

In [ ]:
###################
#PLOTTING FUNCTIONS
fontSettings = {
    "tickFont": 14,
    "labelFont": 16,
    "legendFont": 14,
    "titleFont": 22,
}

In [ ]:
def SetXLimitsDatetime(ax, time_array):
    """
    Ensures datetime x-axis starts and ends exactly at the first and last time values.
    Works for both datetime.datetime and np.datetime64 arrays.
    """
    import numpy as np
    from matplotlib.dates import date2num

    # Convert to Matplotlib’s internal float format if needed
    times = np.asarray(time_array)
    if np.issubdtype(times.dtype, np.datetime64):
        times = date2num(times)
    elif isinstance(times[0], (object,)):
        try:
            times = date2num(times)
        except Exception:
            pass

    ax.set_xlim(times.min(), times.max())
    
def lineplot(axis, time, output, varName, units, color, label=None, linestyle='-', set_ylabel=True):

    axis.plot(time, output.squeeze(), color=color, linestyle=linestyle, label=label)

    if set_ylabel:
        axis.set_ylabel(fr"{varName} ({units})", fontsize=fontSettings["labelFont"])

    axis.set_xlabel("Time", fontsize=fontSettings["labelFont"])

    # axis.grid(True,zorder=-10, linestyle=linestyle)

    plt.setp(axis.get_xticklabels(), rotation=45, ha="right")

    axis.tick_params(axis="both", labelsize=fontSettings["tickFont"])

In [ ]:
def MakeCombinedPlot_1():
    fig = plt.figure(figsize=(8,4))
    gs = GridSpec(1,1,figure=fig)
    
    ax1 = fig.add_subplot(gs[0])
    
    multiplier = 100
    
    # --- Left axis: SCOR ---
    lineplot(ax1, timeList_NSSL, multiplier*SCOR_NSSL,
             varName="Spatial Correlation", units="%", color="blue")
    
    lineplot(ax1, timeList_TEMPO, multiplier*SCOR_TEMPO,
             varName="Spatial Correlation", units="%", color="green")
    
    ax1.set_ylim(-10,100)
    ax1.axhline(0,color='grey',linestyle='--')
    
    # --- Right axis: SRMSE ---
    ax2 = ax1.twinx()
    
    lineplot(ax2, timeList_NSSL, SRMSE_NSSL,
             varName="Spatial RMSE", units="mm", color="blue", linestyle='--')
    
    lineplot(ax2, timeList_TEMPO, SRMSE_TEMPO,
             varName="Spatial RMSE", units="mm", color="green", linestyle='--')
    
    ax2.set_ylim(bottom=0)
    
    # Custom legend
    customLines = [
        Line2D([0], [0], color='blue', lw=2),
        Line2D([0], [0], color='green', lw=2)
    ]
    
    ax1.legend(customLines, ['NSSL', 'TEMPO'], loc='upper right')
    
    # x limits
    ax1.set_xlim(timeList_NSSL[0]- timedelta(hours=1), timeList_NSSL[-1])

    #Adding Title
    fig.suptitle(
        f"{ModelData_NSSL.region} {ModelData_NSSL.case}",
        fontsize=20,
        y=0.96,
        fontweight="bold"
    )
    
    return fig

def MakeCombinedPlot_2():
    fig = plt.figure(figsize=(8,4))
    gs = GridSpec(1,1,figure=fig)
    
    ax1 = fig.add_subplot(gs[0])
    
    
    # --- FSS ---
    lineplot(ax1, timeList_NSSL, FSS_1_NSSL,
             varName="Fractions Skill Score", units="%", color="blue", linestyle='-')
    
    lineplot(ax1, timeList_TEMPO, FSS_1_TEMPO,
             varName="Fractions Skill Score", units="%", color="green", linestyle='-')
    
    lineplot(ax1, timeList_NSSL, FSS_2_NSSL,
             varName="Fractions Skill Score", units="%", color="blue", linestyle='--')
    
    lineplot(ax1, timeList_TEMPO, FSS_2_TEMPO,
             varName="Fractions Skill Score", units="%", color="green", linestyle='--')
    
    ax1.set_ylim(0,100)
    
    # Custom legend
    customLines = [
        Line2D([0], [0], color='blue', lw=2),
        Line2D([0], [0], color='green', lw=2)
    ]
    
    ax1.legend(customLines, ['NSSL', 'TEMPO'], loc='upper right')
    
    # x limits
    ax1.set_xlim(timeList_NSSL[0] - timedelta(hours=1), timeList_NSSL[-1])

    #Adding Title
    fig.suptitle(
        f"{ModelData_NSSL.region} {ModelData_NSSL.case}",
        fontsize=20,
        y=0.96,
        fontweight="bold"
    )
    
    return fig

In [ ]:
def SaveFigure(fig, ModelData1, ModelData2, dpi=300,
               plotType="SCOR_SRMSE"):
    """
    Saves a figure to the appropriate directory based on the models in combinedDict.
    """
    # --- Define output subdirectory and file path ---
    outputSubDirectory = f"{ModelData1.region}_{ModelData1.case}_{ModelData1.spinup_hours}hrs"
    os.makedirs(os.path.join(outputPlottingDirectory, outputSubDirectory), exist_ok=True)

    outputFilePath = os.path.join(
        outputPlottingDirectory,
        outputSubDirectory,
        f"RainfallSpatialMetrics_{plotType}.png"
    )

    # # --- Save figure ---
    # FigurePlotting_Class.SaveUniformFigure(fig, outputFilePath)
    fig.savefig(outputFilePath, dpi=dpi, bbox_inches="tight",pad_inches=0.02)
    plt.close(fig)
    print(f"Saved image: {outputFilePath}")

In [ ]:
###################
#PLOTTING

import matplotlib as mpl
mpl.rcParams['figure.dpi'] = 300

In [ ]:
fig =  MakeCombinedPlot_1()
SaveFigure(fig,ModelData_NSSL,ModelData_TEMPO,
           plotType="RainfallSpatialMetrics")
fig

In [ ]:
fig =  MakeCombinedPlot_2()
SaveFigure(fig,ModelData_NSSL,ModelData_TEMPO,
           plotType="RainfallFSS")
fig

In [ ]:
####################################
#PLOTTING ALL SIMULATIONS

import matplotlib as mpl
mpl.rcParams['figure.dpi'] = 900

plotting = False #keep false when job array is running
plotting = True

In [ ]:
def GetFigureFilePath(region,case,spinup_hours,
                      plotType="RainfallSpatialMetrics",
                      extension="png"):
    # --- Define output subdirectory ---
    inputSubDirectory = f"{region}_{case}_{spinup_hours}hrs"
    load_dir = os.path.join(outputPlottingDirectory, inputSubDirectory)
    # --- File path ---
    inputFilePath = os.path.join(
        load_dir,
        f"{dataType}_{plotType}.{extension}"
    )
    return inputFilePath

def GetFilePaths(plotType):
    caseList = ConsolidateFigures_CLASS.GetCaseList()
    filePaths = []
    for region, case, spinup_hours in caseList:
        filePaths.append(GetFigureFilePath(region,case,spinup_hours,plotType))
    return filePaths

In [ ]:
if plotting:
    sys.path.append(os.path.join(DirectoryManager.mainCodeDirectory,"DataAnalysis"))
    from CLASSES_Plotting import ConsolidateFigures_CLASS

In [ ]:
if plotting:
    plotType = "RainfallSpatialMetrics"
    filePaths = GetFilePaths(plotType=plotType)
    
    fig = ConsolidateFigures_CLASS.AssembleImageGrid(filePaths=filePaths,
                                                     nrows=3,ncols=2,
                                                     figsize=(6, 4),
                                                     wspace=0.01,hspace=0.02,
                                                     dpi=900)
    ConsolidateFigures_CLASS.SaveCombinedFigure(fig,dpi=900, saveDirectory=outputPlottingDirectory,fileName=dataType+f"_{plotType}")

In [ ]:
if plotting:
    plotType = "RainfallFSS"
    filePaths = GetFilePaths(plotType=plotType)
    
    fig = ConsolidateFigures_CLASS.AssembleImageGrid(filePaths=filePaths,
                                                     nrows=3,ncols=2,
                                                     figsize=(6, 5),
                                                     wspace=0.01,hspace=0.02,
                                                     dpi=900)
    ConsolidateFigures_CLASS.SaveCombinedFigure(fig,dpi=900, saveDirectory=outputPlottingDirectory,fileName=dataType+f"_{plotType}")